<a href="https://colab.research.google.com/github/jppeirce/DSC210-Foundations-of-Data-Science/blob/main/Homework/HW08-text_as_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework 8: A Spam Filter That Reads

**DSC 210 Foundations of Data Science**

*Covers Module 12 (text as data).*

```
ASK  ->  GET  ->  EXPLORE  ->  [ MODEL ]  ->  COMMUNICATE
```

References:
- Module 12 course notebook (cleaning and tokens, stop words, TF-IDF, a text classifier, n-grams, search, clustering, next-word models)
- T. A. Almeida, J. M. Gómez Hidalgo, and A. Yamakami (2011), "Contributions to the Study of SMS Spam Filtering: New Collection and Results," *Proceedings of the 11th ACM Symposium on Document Engineering (DocEng '11)*. The SMS Spam Collection is distributed through the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/228/sms+spam+collection).
- `csv_data/sms_spam.csv`: the SMS Spam Collection with 358 messages removed that contain profanity, sexual content, or drug references (see `csv_data/sms_spam_README.txt`).

*Last major revision: 2026-11-28*

## Key Concepts

- Decide how to clean text for a particular task, and test whether the decision helps
- Build, compare, and read TF-IDF + logistic regression classifiers, including their mistakes
- Use search and clustering to investigate a collection of documents
- Build next-word models, and use them to test a classifier
- Explain what a text model can and cannot learn from the data it was given

## Instructions

**Total: 50 points.** Point values are listed with each problem.

- Cells marked `# RUN-TOGETHER` are complete; run them and read the output.
- Cells marked `# FILL-IN` contain blanks written as `____`.
- Cells marked `# YOUR CODE` are yours to write. Each problem says what your code must produce, not how to write it. The Module 12 notebook has the pieces you need; deciding which pieces, and how to change them, is part of the problem.
- Run the cells **in order**. Later problems use variables from earlier ones; the names you must use are given in **bold**.
- Most of the credit rides on your written answers. Quote specific numbers from your output.
- Submit the completed `.ipynb`, with all cells run, to Canvas.

---
## Part 0. Version 5 of the spam filter (2 points)
---

In Homework 6 you built a spam filter from six numbers per email: how many `!` and `$` signs, capital letters, and a few chosen words. The words themselves were gone. This time the filter gets to **read**.

The data is 5,214 real text messages (SMS), each labeled `spam` or `ham` (an ordinary message). The text is exactly as the senders typed it: abbreviations, typos, prices, phone numbers and all.

| Column | Meaning |
| --- | --- |
| `label` | `spam` or `ham` |
| `message` | the text of the message |

**Content note.** These are real messages from strangers. Messages with profanity, sexual content, or drug references were removed before this homework, but the remaining spam is still spam, and some ordinary messages are personal.

In [ ]:
# RUN-TOGETHER  (setup)
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans

sms = pd.read_csv('https://raw.githubusercontent.com/jppeirce/DSC210-Foundations-of-Data-Science/main/csv_data/sms_spam.csv')
print(sms.shape)
print(sms['label'].value_counts())
print()
print('median number of words per message:')
print(sms['message'].str.split().str.len().groupby(sms['label']).median())
print()
for i in [0, 2]:
    print(sms['label'][i], ':', sms['message'][i])

**Problem 0.** (2 points)

(a) What share of the messages are spam? A filter that never flags anything (Homework 6's v0) would score what accuracy?

(b) How do typical spam and ham messages differ in length? Name two other differences you can see in the two printed messages.

**Your answer:**

(a)

(b)

---
## Part A. Cleaning for spam, not for hotels (10 points)
---

### Problem A1. Your own cleaning rule (4 points)

Module 12 cleaned hotel reviews by keeping letters only. Here is what that rule does to a spam message, and how common digits are in each class.

In [ ]:
# RUN-TOGETHER  (Problem A1)
raw = sms['message'][2]
print('original     :', raw)
print('letters only :', re.sub(r'[^a-z\s]', '', raw.lower()).split())
print()
has_digit = sms['message'].str.contains(r'[0-9]')
print('share of messages containing a digit:')
print(has_digit.groupby(sms['label']).mean().round(3))

Write a better rule for this task. Complete the function **`clean_sms`** so that it

1. lowercases the text, and
2. replaces every run of digits with the word ` num ` (with spaces around it), so that `87121` and `08452810075` both become `num`. *(Hint: `re.sub(r'[0-9]+', ' num ', text)`.)*

Then the cell stores the cleaned text in a new column, **`sms['clean']`**, and shows message 2 before and after.

In [ ]:
# YOUR CODE  (Problem A1)
def clean_sms(text):
    # YOUR CODE: lowercase, then replace every run of digits with ' num '

    return text

sms['clean'] = [clean_sms(t) for t in sms['message']]
print(sms['clean'][2])

(a) Name two pieces of message 2 that the letters-only rule destroyed or mangled.

(b) Use the digit shares: why is deleting digits a bad idea for this task, when it was reasonable for hotel reviews?

(c) Your rule turns every number into the same word. Give one reason this could **help** a classifier and one reason it could **hurt**. (Problem B1 will test it.)

**Your answer:**

(a)

(b)

(c)

### Problem A2. Has this text already been cleaned? (3 points)

Module 12 found that someone had removed most stop words from the hotel reviews before publishing them. Find out whether that happened here, and look for any other sign of earlier processing.

Your code must:
1. pick **six** very common English words of your choice and print the percentage of messages that contain each one (as a whole word, ignoring capitals), and
2. count how many messages contain the text `&lt;` and print three of them.

In [ ]:
# YOUR CODE  (Problem A2)

(a) Have stop words been removed from these messages? Quote your numbers.

(b) `&lt;` and `&gt;` are web code for `<` and `>`. What do you think the people who built this dataset replaced with markers like `<#>`, and why? (The documentation does not say, so give your best guess.)

(c) The vectorizer will turn these markers into the tokens `lt` and `gt`. Will those tokens tell the classifier anything about what the *senders* wrote? Explain.

**Your answer:**

(a)

(b)

(c)

### Problem A3. TF-IDF by hand, and by design (3 points)

Three tiny messages, $N = 3$:

$$m_1 = \text{"call me now"} \qquad m_2 = \text{"call now"} \qquad m_3 = \text{"win prize now"}$$

Use $\text{TF} = \dfrac{\text{count}}{\text{tokens in the message}}$ and $\text{IDF} = \ln\left(\dfrac{N}{\text{df}}\right)$ from Module 12, Section 3.2.

(a) Give df and IDF for `call`, `me`, `now`, `prize`, `win`, and the TF-IDF of each token in $m_3$.

(b) Without changing $m_3$, write a **fourth** message that would *lower* the TF-IDF of `win` in $m_3$. Explain why it works, then check it with the cell below (put your message in the blank).

(c) Could any fourth message make the TF-IDF of `call` exactly 0 in $m_1$? Explain.

**Your answer:**

(a)

(b)

(c)

In [ ]:
# FILL-IN  (Problem A3b)
for docs in [['call me now', 'call now', 'win prize now'],
             ['call me now', 'call now', 'win prize now', ____]]:
    vec = TfidfVectorizer(norm=None, smooth_idf=False)
    M = vec.fit_transform(docs)
    table = pd.DataFrame(M.toarray(), columns=vec.get_feature_names_out())
    # scikit-learn adds 1 to every IDF, so subtract 1 to match the hand formula
    idf = pd.Series(vec.idf_ - 1, index=vec.get_feature_names_out())
    print(len(docs), 'messages:  IDF(win) =', round(idf['win'], 3),
          '  TF-IDF(win in m3) by hand =', round(idf['win'] / 3, 3))

---
## Part B. Version 5: build it, read it, break it (14 points)
---

Every filter in Part B uses the same split, so their scores are comparable.

In [ ]:
# RUN-TOGETHER  (the split used for all of Part B)
train, test = train_test_split(sms, test_size=0.25, random_state=210, stratify=sms['label'])
print('training messages:', len(train), '| test messages:', len(test))
print('baseline (always ham) on the test set:', round((test['label'] == 'ham').mean(), 3))

### Problem B1. Four candidate filters (5 points)

Is your cleaning rule from A1 worth it? Is removing stop words worth it? Test both.

Write a function **`evaluate(column, stop_words)`** that, using `train` and `test`:

1. fits a `TfidfVectorizer(max_features=2000, stop_words=stop_words)` on the **training** text in `column` (`'message'` or `'clean'`),
2. fits a `LogisticRegression(max_iter=2000)`,
3. and returns three numbers for the test set: **accuracy**, **spam caught** (the share of test spam that was flagged), and **false alarms** (the number of ham messages flagged as spam).

Then print a small table for all four combinations of `column` in `['message', 'clean']` and `stop_words` in `[None, 'english']`. Module 12, Section 4 has the pieces.

In [ ]:
# YOUR CODE  (Problem B1)

(a) Does removing stop words help or hurt? Check your A2 numbers and suggest which removed words might matter for text messages. (Hint: who is `me` and `my` written by?)

(b) Does your A1 cleaning rule help? Describe the trade-off in **numbers of messages**, not only percentages: how many more spam messages does it catch, and how many more real messages does it flag?

(c) Choose the filter you would ship and say why in one sentence. Call it **version 5**.

**Your answer:**

(a)

(b)

(c)

### Problem B2. What did version 5 learn? (3 points)

Refit your chosen version 5 on the training text and name the fitted pieces **`v5_vec`** and **`v5_model`**, and its test predictions **`v5_pred`**. Then print the 10 words that push hardest toward spam and the 10 that push hardest toward ham. (The classes in alphabetical order are `ham` and `spam`, so a **positive** coefficient pushes toward **spam**.)

In [ ]:
# YOUR CODE  (Problem B2)

(a) Pick three spam words and say in one phrase each what they reveal about how spam is written.

(b) Which word in either list is a **fragile** feature, in the sense of Module 12's `told`: it works on this dataset but could easily fail on messages from somewhere else? Explain.

**Your answer:**

(a)

(b)

### Problem B3. Read the mistakes (4 points)

Print every **spam** message that version 5 let through, and every **ham** message it flagged. (Use `test` and `v5_pred`.)

In [ ]:
# YOUR CODE  (Problem B3)

(a) Read the spam that got through. Describe two kinds of spam the filter misses, with an example of each.

(b) Read the real messages that were flagged. What made them look like spam?

(c) Suggest one change, to the data, the cleaning, or the model, that might catch some of the missed spam. You do not have to test it.

**Your answer:**

(a)

(b)

(c)

### Problem B4. Do word pairs help? (2 points)

Add bigrams (`ngram_range=(1, 2)`) to your version 5 and report the test accuracy, spam caught, and false alarms. Then find one bigram among the 10 strongest in either direction whose meaning is different from its first word alone, and explain.

In [ ]:
# YOUR CODE  (Problem B4)

**Your answer:**

---
## Part C. Investigating the collection (12 points)
---

### Problem C1. Which message does search return first? (3 points)

A user searches for `"free call"`. Count each text over the vocabulary `free`, `call`, `now`, `me`:

| text | free | call | now | me |
| --- | --- | --- | --- | --- |
| query $q$: "free call" | 1 | 1 | 0 | 0 |
| $u$: "free call now free call now" | 2 | 2 | 2 | 0 |
| $w$: "call me" | 0 | 1 | 0 | 1 |

(a) Compute the Euclidean distances $d(q, u)$ and $d(q, w)$. Which message would a distance-based search return first?

(b) Rescale $q$, $u$, and $w$ to length 1 (Module 12, Section 6.2) and recompute both distances. Which is returned first now?

(c) Which result better matches what the user was looking for, and why?

**Your answer:**

(a)

(b)

(c)

### Problem C2. Investigate one kind of scam (4 points)

Pick **one** kind of spam you noticed in B3 or Part 0, for example prize draws, ringtone subscriptions, "someone has asked our dating service to contact you", or "you have a new voicemail, call...".

Your code must:
1. build a search engine over **all** the messages, as in Module 12, Class Example 12.2, that prints the label and text of the top matches;
2. run **two different queries** for your kind of scam and show the top 5 results of each;
3. pick one word or short phrase that marks your kind of scam, and count how many messages contain it and how many of those are labeled spam.

In [ ]:
# YOUR CODE  (Problem C2)

(a) Which kind of scam did you choose? Did your two queries find it? Did either query return anything that is *not* that scam?

(b) Report your count. Is your marker word a good spam detector on its own? What would it miss?

**Your answer:**

(a)

(b)

### Problem C3. Can clustering find the spam without labels? (5 points)

Suppose the labels did not exist. Could k-means still put most spam in one cluster?

Using Module 12, Class Example 12.3 as a starting point, cluster the `message` column and, for each cluster, print its **size**, its **share of spam**, and its **top 10 words**. Then experiment with the settings (`k`, `seed`, `max_df`, and whether to remove stop words). Your goal: a cluster that is **at least 80% spam** and contains **at least 200 messages**.

Keep the cell set to your best settings when you submit, and record in your answer at least **three** settings you tried and what happened.

In [ ]:
# YOUR CODE  (Problem C3)

(a) Record at least three settings you tried, and for each the size and spam share of the most spam-heavy cluster.

(b) Describe your best result. Did it meet the goal? What are the other clusters about? Name at least one cluster that formed for a reason other than what the messages mean.

(c) Change only the seed of your best setting. Does your spam cluster survive? What does this say about using clustering, rather than the classifier from Part B, to find spam?

**Your answer:**

(a)

(b)

(c)

---
## Part D. Fake spam (8 points)
---

### Problem D1. Two message-writing machines (3 points)

Using the functions from Module 12, Section 8 (`tokenize`, `count_next_words`, and `generate`), build a bigram model **`spam_model`** from the training spam (`train` rows labeled spam) and a bigram model **`ham_model`** from the training ham. Set `rng = np.random.default_rng(210)` before generating, and print three messages from each model.

In [ ]:
# YOUR CODE  (Problem D1)

Without looking anything up, describe how the two models' messages differ. Name the words that give each one away.

**Your answer:**

### Problem D2. Can version 5 catch fake spam? (5 points)

Generate **200** messages from each model, clean them with `clean_sms`, and run them through version 5 (`v5_vec` and `v5_model`). Print the share of fake spam that version 5 flags, the share of fake ham that it flags, and **ten** fake spam messages that it did *not* flag.

In [ ]:
# YOUR CODE  (Problem D2)

(a) Report the two shares. Is fake spam easier or harder for version 5 to catch than real test spam?

(b) Read the fake spam that got through. What do many of them have in common? Explain, using how the bigram model decides when a message ends (Module 12, Section 8) and what version 5 looks for.

(c) A real spammer wants to get past version 5. Give two reasons this experiment does **not** tell us how well version 5 would hold up against one. (Hint: where did both the generator and the filter learn their words? And look again at B3.)

**Your answer:**

(a)

(b)

(c)

---
## Part E. Report to the phone company (4 points)
---

A phone company wants to use your version 5. Write 4 to 6 sentences for a manager who has never trained a model. Say:

- whether you would let it **delete** suspected spam or move it to a **spam folder**, using your numbers from B1;
- one feature of the data (from Parts A to C) that might make it work worse on the company's own customers' messages;
- one change to the training data you would ask for before going live.

**Your answer:**